In [1]:
from backend.utils.lossy_electric_conductor import *
from backend.utils.file_path import setup_save_file_paths
from backend.utils.gmsh_function import *

from backend.rwg.rwg1 import *
from backend.rwg.rwg2 import *
from backend.utils.impmet import *

In [2]:
# Constants for Copper (Normal Conductor example)
freq = 1e9  # 1 GHz
sigma_copper = 5.8e7  # S/m

In [3]:
# Case 1: Standard Copper (mu_r is 1.0 by default)
zs_cu = calculate_normal_surface_impedance(freq, sigma_copper)
print(f"Copper (Non-magnetic, mu_r=1): {zs_cu:.4f} Ohms")

Copper (Non-magnetic, mu_r=1): 0.0083+0.0083j Ohms


In [4]:
# Case 2: Ferromagnetic material (e.g., Nickel or Steel)
# Let's assume mu_r = 100 for a specific steel alloy
mu_r_steel = 100
zs_steel = calculate_normal_surface_impedance(freq, 1e7, mu_r=mu_r_steel)
print(f"Steel (Magnetic, mu_r=100): {zs_steel:.4f} Ohms")

Steel (Magnetic, mu_r=100): 0.1987+0.1987j Ohms


In [5]:
# Constants for a Superconductor example
lambda_l = 50e-9  # 50 nm
zs_super = calculate_superconductor_surface_impedance(freq, lambda_l)
print(f"\nSuperconductor Zs: {zs_super} Ohms")


Superconductor Zs: 0.00039489819655623697j Ohms


In [6]:
name = "strip_copper"
path = setup_save_file_paths(name)
mm = 1e-3
m = 1
longueur = 2 * m
largeur = 2 * m
initial_mesh_size = 10 * m

# Coding the Nice CubeSat geometry
gmsh.initialize()

gmsh.model.add("Strip_copper")

setup_performance_config()

strip_antenna = gmsh.model.occ.addRectangle(-longueur / 2, -largeur / 2, 0, longueur, largeur)

generate_and_save_mesh(path, initial_mesh_size)

# gmsh.fltk.run()

gmsh.finalize()

extract_msh_to_mat(path.msh, path.mat)
    
# Load the mesh file
p, t = load_mesh_file(path.mat)

# Define points and triangles from the mesh
points = Points(p)
triangles = Triangles(t)

# Compute geometric properties (areas, centers)
triangles.calculate_triangles_area_and_center(points)

# Define edges and compute their lengths
edges = triangles.get_edges()

filter_complexes_jonctions(points, triangles, edges)  # Filter complex junctions to simplify the mesh structure

edges.compute_edges_length(points)

# Save processed mesh data
DataManager_rwg1.save_data(path, points, triangles, edges)

# Definition and calculation of barycentric triangles
barycentric_triangles = Barycentric_triangle()
barycentric_triangles.calculate_barycentric_center(points, triangles)

# Calculation of RHO vectors for edges
vecteurs_rho = Vecteurs_Rho()
vecteurs_rho.calculate_vecteurs_rho(points, triangles, edges, barycentric_triangles)

# Save barycentric triangles and RHO vectors data
DataManager_rwg2.save_data(path, barycentric_triangles, vecteurs_rho)

In [7]:
points, triangles, edges, _, vecteurs_rho = DataManager_rwg2.load_data(path.mat_mesh2)

G2 = rwg_gram_matrix(edges, triangles, vecteurs_rho)
print(G2)

Starting vectorized G calculation for 4 triangles...
  Processed 0/4 triangles...
[[ 1.30864198e+00 -3.45679012e-01  0.00000000e+00  4.62592927e-18]
 [-3.45679012e-01  1.30864198e+00 -6.16790569e-18  0.00000000e+00]
 [ 0.00000000e+00 -6.16790569e-18  1.30864198e+00  3.45679012e-01]
 [ 4.62592927e-18  0.00000000e+00  3.45679012e-01  1.30864198e+00]]
